# CatBoost Edge AI Deployment

This notebook prepares the selected CatBoost occupancy classification model for deployment on an edge device (Raspberry Pi).

The model predicts the occupancy level of a room 30 minutes ahead using building sensor data.

In [1]:
from pathlib import Path
import json
import time
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score

from catboost import CatBoostClassifier
import joblib

RANDOM_STATE = 42

DATA_PATH = Path("../data/processed/model_table_5min.csv")
REPORT_DIR = Path("../reports/occupancy_classification")

CLASS_LABELS = [0, 1, 2, 3, 4]
CLASS_NAMES = ["Empty", "Low", "Moderate", "High", "Very high"]

print("Working directory:", Path.cwd())
print("Dataset path:", DATA_PATH.resolve())

Working directory: c:\Users\leena\OneDrive\Desktop\adaptive-district-backbone\notebooks
Dataset path: C:\Users\leena\OneDrive\Desktop\adaptive-district-backbone\data\processed\model_table_5min.csv


In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

data = pd.read_csv(DATA_PATH)

print("Dataset shape:", data.shape)
print("Number of days:", data["DayId"].nunique())
print("Rooms:", sorted(data["room"].dropna().unique()))

display(data.head())

Dataset shape: (36432, 58)
Number of days: 44
Rooms: [np.int64(1), np.int64(2), np.int64(3)]


,DayId,bin,occ,cam1_fw,cam1_bw,cam2_fw,cam2_bw,co2,temperature,humidity,...,humidity_diff,cam1_fw_last15,cam1_bw_last15,cam2_fw_last15,cam2_bw_last15,target,capacity,event_day,occ_ratio,is_lecture
0,0,6,0.0,0.0,0.0,0.0,0.0,482.880005,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1
1,0,7,0.0,0.0,0.0,0.0,0.0,480.000000,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1
2,0,8,0.0,0.0,0.0,0.0,0.0,475.839996,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1
3,0,9,0.0,0.0,0.0,0.0,0.0,478.720001,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1
4,0,10,0.0,0.0,0.0,0.0,0.0,473.920013,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1


In [3]:
occupancy_ratio_future = data["target"] / data["capacity"]

data["target_class"] = pd.cut(
    occupancy_ratio_future,
    bins=[-0.001, 0, 0.25, 0.50, 0.75, np.inf],
    labels=CLASS_LABELS,
    include_lowest=True
).astype("Int64")

data = data.dropna(
    subset=["target_class", "DayId", "capacity"]
).copy()

data["target_class"] = data["target_class"].astype(int)

class_distribution = (
    data["target_class"]
    .value_counts()
    .reindex(CLASS_LABELS, fill_value=0)
)

class_distribution.index = CLASS_NAMES

display(class_distribution.to_frame("Number of rows"))

,Number of rows
Empty,16523
Low,13301
Moderate,3983
High,1883
Very high,742


In [4]:
leakage_or_id_cols = {
    "target", "target_class", "event_day", "DayId",
    "occ_future", "target_future", "future_occupancy"
}

feature_cols = [
    c for c in data.columns
    if c not in leakage_or_id_cols
    and pd.api.types.is_numeric_dtype(data[c])
]

feature_cols = [
    c for c in feature_cols
    if data[c].notna().any()
]

X = data[feature_cols].copy()
y = data["target_class"].astype(int)
groups = data["DayId"]

print("Number of features:", len(feature_cols))
print("\nFeature names:")
print(feature_cols)

print("\nFeature matrix:", X.shape)
print("Target:", y.shape)

Number of features: 55

Feature names:
['bin', 'occ', 'cam1_fw', 'cam1_bw', 'cam2_fw', 'cam2_bw', 'co2', 'temperature', 'humidity', 'illuminance', 'vav', 'co2_age', 'temperature_age', 'humidity_age', 'illuminance_age', 'vav_age', 'room', 'workday', 'month', 'hour_sin', 'hour_cos', 'occ_lag1', 'occ_lag2', 'occ_lag3', 'occ_lag6', 'co2_lag1', 'co2_lag2', 'co2_lag3', 'co2_lag6', 'temperature_lag1', 'temperature_lag2', 'temperature_lag3', 'temperature_lag6', 'humidity_lag1', 'humidity_lag2', 'humidity_lag3', 'humidity_lag6', 'illuminance_lag1', 'illuminance_lag2', 'illuminance_lag3', 'illuminance_lag6', 'vav_lag1', 'vav_lag2', 'vav_lag3', 'vav_lag6', 'co2_diff', 'temperature_diff', 'humidity_diff', 'cam1_fw_last15', 'cam1_bw_last15', 'cam2_fw_last15', 'cam2_bw_last15', 'capacity', 'occ_ratio', 'is_lecture']

Feature matrix: (36432, 55)
Target: (36432,)


In [5]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

assert set(groups_train).isdisjoint(set(groups_test))

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training days:", groups_train.nunique())
print("Testing days:", groups_test.nunique())

Training rows: 28980
Testing rows: 7452
Training days: 35
Testing days: 9


In [6]:
catboost_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    CatBoostClassifier(
        iterations=350,
        depth=5,
        learning_rate=0.04,
        loss_function="MultiClass",
        verbose=False,
        random_seed=RANDOM_STATE,
        auto_class_weights="Balanced"
    )
)

start_time = time.perf_counter()

catboost_pipeline.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Training time: 8.77 seconds


In [7]:
predictions = catboost_pipeline.predict(X_test)
predictions = np.asarray(predictions).reshape(-1).astype(int)

print(classification_report(
    y_test,
    predictions,
    labels=CLASS_LABELS,
    target_names=CLASS_NAMES,
    zero_division=0
))

print("Macro F1:", round(
    f1_score(y_test, predictions, average="macro"), 4
))

              precision    recall  f1-score   support

       Empty       0.92      0.91      0.91      3067
         Low       0.87      0.81      0.84      2874
    Moderate       0.60      0.67      0.64       761
        High       0.57      0.70      0.63       550
   Very high       0.59      0.55      0.57       200

    accuracy                           0.82      7452
   macro avg       0.71      0.73      0.72      7452
weighted avg       0.83      0.82      0.83      7452

Macro F1: 0.7172


In [8]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    catboost_pipeline,
    REPORT_DIR / "catboost_pipeline.joblib"
)

with open(REPORT_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=4)

class_metadata = {
    "0": "Empty",
    "1": "Low",
    "2": "Moderate",
    "3": "High",
    "4": "Very high"
}

with open(REPORT_DIR / "class_labels.json", "w") as f:
    json.dump(class_metadata, f, indent=4)

In [9]:
loaded_model = joblib.load(
    REPORT_DIR / "catboost_pipeline.joblib"
)

with open(REPORT_DIR / "feature_columns.json", "r") as f:
    loaded_features = json.load(f)

with open(REPORT_DIR / "class_labels.json", "r") as f:
    loaded_classes = json.load(f)

print("Model loaded successfully")
print("Number of features:", len(loaded_features))
print("Classes:", loaded_classes)

Model loaded successfully
Number of features: 55
Classes: {'0': 'Empty', '1': 'Low', '2': 'Moderate', '3': 'High', '4': 'Very high'}


In [10]:
test_predictions = np.asarray(
    loaded_model.predict(X_test)
).reshape(-1).astype(int)

print(classification_report(
    y_test,
    test_predictions,
    labels=CLASS_LABELS,
    target_names=CLASS_NAMES,
    zero_division=0
))

assert np.array_equal(predictions, test_predictions)

print("Predictions match the original model!")

              precision    recall  f1-score   support

       Empty       0.92      0.91      0.91      3067
         Low       0.87      0.81      0.84      2874
    Moderate       0.60      0.67      0.64       761
        High       0.57      0.70      0.63       550
   Very high       0.59      0.55      0.57       200

    accuracy                           0.82      7452
   macro avg       0.71      0.73      0.72      7452
weighted avg       0.83      0.82      0.83      7452

Predictions match the original model!


In [13]:
import importlib
import edge_inference

importlib.reload(edge_inference)

result = edge_inference.predict_occupancy(sample_input)

print(result)

{'class_id': 0, 'occupancy_level': 'Empty'}


In [14]:
import sys

sys.path.insert(0, str(Path.cwd().parent))

from edge_inference import predict_occupancy

sample_input = X_test.iloc[0].to_dict()

result = predict_occupancy(sample_input)

print(result)

{'class_id': 0, 'occupancy_level': 'Empty'}


In [15]:
import time
import numpy as np

sample_input = X_test.iloc[0].to_dict()

# Warm-up
for _ in range(10):
    predict_occupancy(sample_input)

# Benchmark
times = []

for _ in range(1000):
    start = time.perf_counter()
    predict_occupancy(sample_input)
    times.append((time.perf_counter() - start) * 1000)

print("Average latency:", round(np.mean(times), 3), "ms")
print("Median latency:", round(np.median(times), 3), "ms")
print("95th percentile:", round(np.percentile(times, 95), 3), "ms")

Average latency: 3.438 ms
Median latency: 2.797 ms
95th percentile: 5.132 ms


In [16]:
model_path = REPORT_DIR / "catboost_pipeline.joblib"

model_size_mb = model_path.stat().st_size / (1024 ** 2)

print("Model size:", round(model_size_mb, 2), "MB")

Model size: 0.56 MB
